# Adding random field audio to my negatives dataset
1. Generate list of random field audio files from rloc2025a, one from each recorder. 
2. Create .pkl file with each hour split into 1.5 second chunks, matching the df format of the train anf val dfs. 
3. Integrate that field_negatives.pkl into my existing dataset.

In [1]:
import os
import random
from pathlib import Path
import pandas as pd

In [2]:

def randomly_select_wav_files(root_dir):
    """
    Randomly select one .wav file from each subdirectory and create 1.5 second clips
    """
    root_path = Path(root_dir)
    selected_files = []
    
    # Find all subdirectories
    subdirs = [d for d in root_path.iterdir() if d.is_dir()]
    
    print(f"Found {len(subdirs)} subdirectories in {root_dir}")
    
    for subdir in subdirs:
        # Find all .wav files in this subdirectory
        wav_files = list(subdir.glob("*.WAV"))
        
        if wav_files:
            # Randomly select one file
            selected_file = random.choice(wav_files)
            
            # Create 1.5 second clips from 15-minute file
            file_duration = 15 * 60  # 15 minutes = 900 seconds
            clip_duration = 1.0  # 1 seconds
            
            # Calculate number of clips
            num_clips = int(file_duration / clip_duration)
            
            for clip_num in range(num_clips):
                start_time = clip_num * clip_duration
                end_time = start_time + clip_duration
                
                selected_files.append({
                    'subdirectory': subdir.name,
                    'selected_file': selected_file.name,
                    'full_path': str(selected_file),
                    'total_wav_files': len(wav_files),
                    'start_time': start_time,
                    'end_time': end_time,
                    'field_data': 1,
                    'clip_number': clip_num + 1
                })
            
            print(f"{subdir.name}: Selected {selected_file.name} from {len(wav_files)} .WAV files, created {num_clips} clips")
        else:
            print(f"{subdir.name}: No .WAV files found")
    
    return selected_files

In [3]:

# Set random seed for reproducibility
random.seed(42)

# Select files
root_directory = "/media/kiwi/datasets/unfinalized/rloc2025a"
selected_files = randomly_select_wav_files(root_directory)

print(f"\nTotal selected files: {len(selected_files)}")

Found 36 subdirectories in /media/kiwi/datasets/unfinalized/rloc2025a
MSD-3593: Selected 20250509_120000.WAV from 203 .WAV files, created 900 clips
MSD-2774: Selected 20250518_100000.WAV from 218 .WAV files, created 900 clips
MSD-2809: Selected 20250517_070000.WAV from 215 .WAV files, created 900 clips
MSD-0260: Selected 20250530_070000.WAV from 216 .WAV files, created 900 clips
MSD-3792: Selected 20250519_000000.WAV from 207 .WAV files, created 900 clips
MSD-2723: Selected 20250510_110000.WAV from 217 .WAV files, created 900 clips
MSD-3737: Selected 20250519_030000.WAV from 212 .WAV files, created 900 clips
MSD-3608: No .WAV files found
MSD-3579: Selected 20250528_110000.WAV from 211 .WAV files, created 900 clips
MSD-3513: Selected 20250509_000000.WAV from 218 .WAV files, created 900 clips
MSD-3456: Selected 20250525_010000.WAV from 212 .WAV files, created 900 clips
MSD-3632: Selected 20250509_120000.WAV from 215 .WAV files, created 900 clips
MSD-3651: Selected 20250525_000000.WAV fro

In [4]:

# Convert to DataFrame for easier viewing and manipulation
if selected_files:
    df_selected = pd.DataFrame(selected_files)
    print("\nSelected files summary:")
    print(df_selected)
    
    # Save to CSV for reference
    BASE_OUTPUT = Path("/media/auk/projects/gak76/vira_beg_outputs")

    FIELD_NEG_DIR = BASE_OUTPUT / "training_data" / "field_negatives"
    FIELD_NEG_DIR.mkdir(parents=True, exist_ok=True)

    output_file = FIELD_NEG_DIR / "selected_field_negatives_gk.csv"
    df_selected.to_csv(output_file, index=False)
    print(f"\nSaved selected files to {output_file}")
else:
    print("No files were selected")


Selected files summary:
      subdirectory        selected_file  \
0         MSD-3593  20250509_120000.WAV   
1         MSD-3593  20250509_120000.WAV   
2         MSD-3593  20250509_120000.WAV   
3         MSD-3593  20250509_120000.WAV   
4         MSD-3593  20250509_120000.WAV   
...            ...                  ...   
29695     MSD-2545  20250529_030000.WAV   
29696     MSD-2545  20250529_030000.WAV   
29697     MSD-2545  20250529_030000.WAV   
29698     MSD-2545  20250529_030000.WAV   
29699     MSD-2545  20250529_030000.WAV   

                                               full_path  total_wav_files  \
0      /media/kiwi/datasets/unfinalized/rloc2025a/MSD...              203   
1      /media/kiwi/datasets/unfinalized/rloc2025a/MSD...              203   
2      /media/kiwi/datasets/unfinalized/rloc2025a/MSD...              203   
3      /media/kiwi/datasets/unfinalized/rloc2025a/MSD...              203   
4      /media/kiwi/datasets/unfinalized/rloc2025a/MSD...              203

In [5]:
# Rename and clean up the DataFrame
df_selected = df_selected.rename(columns={'full_path': 'file'})
df_selected = df_selected.drop(columns=['clip_number', 'subdirectory', 'total_wav_files', 'selected_file'])
df_selected = df_selected.reset_index(drop=True)
print("Cleaned DataFrame")

Cleaned DataFrame


In [6]:
df_selected.head()

,file,start_time,end_time,field_data
0,/media/kiwi/datasets/unfinalized/rloc2025a/MSD...,0.0,1.0,1
1,/media/kiwi/datasets/unfinalized/rloc2025a/MSD...,1.0,2.0,1
2,/media/kiwi/datasets/unfinalized/rloc2025a/MSD...,2.0,3.0,1
3,/media/kiwi/datasets/unfinalized/rloc2025a/MSD...,3.0,4.0,1
4,/media/kiwi/datasets/unfinalized/rloc2025a/MSD...,4.0,5.0,1


In [7]:
# Save the cleaned DataFrame as a pickle file
#pkl_output_file = "/home/brg226/projects/vira_beg/training_data/field_negatives/field_negatives.pkl"
pkl_output_file = FIELD_NEG_DIR / "field_negatives_gk.pkl"
df_selected.to_pickle(pkl_output_file)
print(f"Saved cleaned DataFrame to {pkl_output_file}")

print(f"\nFinal DataFrame shape: {df_selected.shape}")
print(f"Columns: {df_selected.columns.tolist()}")

Saved cleaned DataFrame to /media/auk/projects/gak76/vira_beg_outputs/training_data/field_negatives/field_negatives_gk.pkl

Final DataFrame shape: (29700, 4)
Columns: ['file', 'start_time', 'end_time', 'field_data']
